In [3]:
import os
import dotenv
import wandb
import tensorflow as tf
import utils


In [4]:
# --- Dynamic GPU/CPU Selection ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPUs detected: {[gpu.name for gpu in gpus]}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"Could not set memory growth for GPUs: {e}")
else:
    print("No GPUs detected. Using CPU only.")
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# --- W&B Setup ---
dotenv.load_dotenv()
key = os.environ.get("WANDB_API_KEY")

if key:
    print("Logging into Weights & Biases...")
    wandb.login(key=key)
else:
    print("WARNING: WANDB_API_KEY not found in .env. Logging in anonymously.")
    wandb.login()

wandb.init(project="animal-faces-classification")

# --- Training Configuration ---
NUM_EPOCHS = 20
LEARNING_RATE = 0.00001

GPUs detected: ['/physical_device:GPU:0']


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pshashid (pshashid-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
if __name__ == "__main__":
    try:
        print("--- Calling Data Preparation Pipeline from utils.py ---")
        train_gen, val_gen = utils.main_prep()

        print("\n--- Training Single ResNet50 Model ---")
        single_resnet = utils.train_model(train_gen, val_gen, architecture="ResNet50", epochs=NUM_EPOCHS, lr=LEARNING_RATE)
        print("\n--- Training Homogeneous Ensemble (ResNet50 x3) ---")
        models_homo, preds_homo, acc_homo = utils.homogeneous_ensemble(train_gen, val_gen, num_models=3, epochs=NUM_EPOCHS, lr=LEARNING_RATE)

        print("\n--- Training Heterogeneous Ensemble (ResNet50 + EfficientNetB0 + MobileNetV2) ---")
        architectures = ["ResNet50", "EfficientNetB0", "MobileNetV2"]
        models_hetero, preds_hetero, acc_hetero = utils.heterogeneous_ensemble(train_gen, val_gen, architectures, epochs=NUM_EPOCHS, lr=LEARNING_RATE)
    except Exception as e:
        print(f"\nAn error occurred during pipeline execution: {e}")
    finally:
        if wandb.run:
            wandb.finish()

--- Calling Data Preparation Pipeline from utils.py ---
--- Starting Data Preparation Pipeline ---
Using Colab cache for faster access to the 'animal-faces' dataset.
Path to dataset files: /kaggle/input/animal-faces


Train data collected. Samples: 14630


,image_path,label
0,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
1,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
2,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
3,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
4,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat



Validation data collected. Samples: 1500


,image_path,label
0,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat
1,/kaggle/input/animal-faces/afhq/val/cat/flickr...,cat
2,/kaggle/input/animal-faces/afhq/val/cat/flickr...,cat
3,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat
4,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat



Setting up ImageDataGenerators (including Data Augmentation)...
Found 14630 validated image filenames belonging to 3 classes.
Found 1500 validated image filenames belonging to 3 classes.
Data Augmentation applied. Generators created successfully.

--- Verification of Data Generators ---

[A] Location of Source Images (first 2 paths from train_df):
  - /kaggle/input/animal-faces/afhq/train/cat/pixabay_cat_000354.jpg
  - /kaggle/input/animal-faces/afhq/train/cat/pixabay_cat_002763.jpg
Total Source Images on Disk: 16130

[B] Generated Samples (Source Count) per Generator:
  - Training Samples (train_generator.samples): 14630
  - Validation Samples (val_generator.samples): 1500
  - Batch Size: 256

[B] Batches per Epoch (The number of times augmentation runs per epoch):
  - Training Steps per Epoch: 58
  - Validation Steps per Epoch: 6
--- Verification Complete ---

 Data Preparation Complete. Generators are ready for Transfer Learning.

--- Training Single ResNet50 Model ---

--- Trainin

...

MobileNetV2_Hetero_val_accuracy,▁
epoch/accuracy,▁▇▇█████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▅▆▆▇▇▇▇▇███████████
epoch/val_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
MobileNetV2_Hetero_val_accuracy,0.99467
epoch/accuracy,0.99768
epoch/epoch,19
epoch/learning_rate,1e-05



--- Evaluating Heterogeneous_Ensemble Ensemble ---


6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step
Heterogeneous_Ensemble Validation Accuracy: 0.9893


heterogeneous_ensemble_val_accuracy,▁
heterogeneous_ensemble_val_accuracy,0.98933



Heterogeneous Ensemble Accuracy: 0.9893
